## Basic trip moving cargo
In this notebook, we set up a basic simulation where one vessel moves over a 1D network path consisting of one edge. We add the HasContainer mixin to both the nodes of the network and the vessel. We created a simple customized mission, where the vessel sails back and forth loading/unloading until the destination site is full.

#### 0. Import libraries

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
from pyproj import Geod
from shapely.geometry import Point

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart
import opentnsim.graph.mixins as graph_module

# package(s) needed for inspecting the output
import pandas as pd
import numpy as np



print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 2.2.2.dev24+g815986113.d20251127


#### 1. Define object classes

In [2]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        opentnsim.core.Identifiable, # allows to give the object a name and a random ID,
        opentnsim.core.Movable,      # allows the object to move, with a fixed speed, while logging this activity
        opentnsim.core.HasContainer, # allows the object to contain cargo
    ), 
    {}
)

In [3]:
# make your preferred Node class out of available mix-ins.
Node = type(
    "Node", 
    (
        opentnsim.core.Identifiable, # allows to give the object a name and a random ID, 
        opentnsim.core.Locatable,    # allows the object to have a location
        opentnsim.core.HasContainer, # allows the object to contain cargo
    ),
    {}
)

#### 2. Create graph
Next we create a network (a graph) along which the vessel can move. For this case we create a single edge of 100 km exactly.

In [4]:
# initialize geodetic calculator with WGS84 ellipsoid
geod = Geod(ellps="WGS84")

In [5]:
# starting point (longitude, latitude)
lon0, lat0 = 0, 0

# compute the other point 100 km East from Point 0
lon1, lat1, _ = geod.fwd(lon0, lat0, 90, 100000) # East from point 1

# define nodes with their geographic coordinates, and capacity and level
node_info = {
    "0": {'position': (lon0, lat0), 'capacity': 100, 'level': 100},
    "1": {'position': (lon1, lat1), 'capacity': 100, 'level': 0},
}

In [6]:
# create list of edges
edges = [("0", "1"), ("1", "0")] # bi-directional edge

In [7]:
# create a directed graph
FG = nx.DiGraph()

# add nodes
for name, node in node_info.items():
    FG.add_node(name, geometry=Point(node['position'][0], node['position'][1]))

# add edges
for edge in edges:
    FG.add_edge(edge[0], edge[1], weight=1)

In [8]:
graph_module.plot_graph(FG)

#### 3. Run simulation

In [13]:
# start simpy environment
simulation_start = datetime.datetime(2024, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

# create sites from dict  
nodes = []
for name, node in node_info.items():
    data_node = {
        "env": env,
        "name": name,
        "geometry": Point(node['position'][0], node['position'][1]), 
        "capacity": node['capacity'],
        "level": node['level'],
    }
    nodes.append(Node(**data_node))
    
# add sites to graph
for node in nodes:
    FG.add_node(node.name, geometry=node.geometry, site=node)

# add graph to environment
env.graph = FG

# define initial route from origin to destination (always use env.graph after assignment)
initial_route = nx.dijkstra_path(env.graph, '0', '1')

# create vessel from dict 
data_vessel = {
    "env": env,                                   # needed for simpy simulation, left empty for now
    "name": "Vessel",                             # required by Identifiable
    "geometry": env.graph.nodes['0']['geometry'], # required by Locatable, left empty for now
    "route": initial_route,                       # required by Routeable, define route once here
    "v": 1,                                       # required by Movable, 1 m/s to check if the distance is covered in the expected time
    "capacity": 30,                               # required by HasContainer
    "level": 0,                                   # required by HasContainer
}  
vessel = Vessel(**data_vessel)

# start the simulation and execute a cargo mission
# you can set different loading and unloading durations per unit of load if desired
env.process(vessel.execute_vessel_mission(mission_type="cargo", 
                                          origin='0', 
                                          destination='1',
                                          duration_per_unit_load=1000,
                                          duration_per_unit_unload=500))
env.run()

#### 4. Inspect output
We can now inspect  the simulation output by inspecting the _vessel.logbook_. Note that the _Log_ mix-in was included when we added _Movable_. The _vessel.logbook_ keeps track of the moving activities of the vessel. For each discrete event OpenTNSim logs an event message, the start/stop time and the location. The _vessel.logbook_ is of type dict. For convenient inspection it can be loaded into a Pandas dataframe. 

In [14]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  


display(df)

'Vessel' logbook data:


,Message,Timestamp,Value,Geometry
0,Loading from node 0 start,2024-01-01 00:00:00,0.0,POINT (0 0)
1,Loading from node 0 stop,2024-01-01 08:20:00,0.0,POINT (0 0)
2,Sailing from node 0 to node 1 start,2024-01-01 08:20:00,0.0,POINT (0 0)
3,Sailing from node 0 to node 1 stop,2024-01-02 12:06:40,100000.0,POINT (0.8983152841195217 0)
4,Unloading to node 1 start,2024-01-02 12:06:40,0.0,POINT (0.8983152841195217 0)
5,Unloading to node 1 stop,2024-01-02 16:16:40,0.0,POINT (0.8983152841195217 0)
6,Sailing from node 1 to node 0 start,2024-01-02 16:16:40,100000.0,POINT (0.8983152841195217 0)
7,Sailing from node 1 to node 0 stop,2024-01-03 20:03:20,200000.0,POINT (0 0)
8,Loading from node 0 start,2024-01-03 20:03:20,0.0,POINT (0 0)
9,Loading from node 0 stop,2024-01-04 04:23:20,0.0,POINT (0 0)


The inspection of the logbook data shows that Vessel moved from its origin (*Node 0*) to its destination (*Node 1*). The print statements show that the length of the route from *Node 0* to *Node 1* is exactly 100 km. At a given speed of 1 m/s the trip duration should be exactly 100000 seconds, as is indeed shown to be the case.

In [15]:
df_eventtable = logbook2eventtable([vessel])
df_eventtable

,object id,object name,activity name,start location,stop location,start time,stop time,distance (m),duration (s)
0,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Loading from node 0,POINT (0 0),POINT (0 0),2024-01-01 00:00:00,2024-01-01 08:20:00,0.0,30000.0
1,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Sailing from node 0 to node 1,POINT (0 0),POINT (0.8983152841195217 0),2024-01-01 08:20:00,2024-01-02 12:06:40,100000.0,100000.0
2,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Unloading to node 1,POINT (0.8983152841195217 0),POINT (0.8983152841195217 0),2024-01-02 12:06:40,2024-01-02 16:16:40,0.0,15000.0
3,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Sailing from node 1 to node 0,POINT (0.8983152841195217 0),POINT (0 0),2024-01-02 16:16:40,2024-01-03 20:03:20,100000.0,100000.0
4,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Loading from node 0,POINT (0 0),POINT (0 0),2024-01-03 20:03:20,2024-01-04 04:23:20,0.0,30000.0
5,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Sailing from node 0 to node 1,POINT (0 0),POINT (0.8983152841195217 0),2024-01-04 04:23:20,2024-01-05 08:10:00,100000.0,100000.0
6,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Unloading to node 1,POINT (0.8983152841195217 0),POINT (0.8983152841195217 0),2024-01-05 08:10:00,2024-01-05 12:20:00,0.0,15000.0
7,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Sailing from node 1 to node 0,POINT (0.8983152841195217 0),POINT (0 0),2024-01-05 12:20:00,2024-01-06 16:06:40,100000.0,100000.0
8,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Loading from node 0,POINT (0 0),POINT (0 0),2024-01-06 16:06:40,2024-01-07 00:26:40,0.0,30000.0
9,1aa0124c-dab2-4b2b-90f5-f1b6e23517dd,Vessel,Sailing from node 0 to node 1,POINT (0 0),POINT (0.8983152841195217 0),2024-01-07 00:26:40,2024-01-08 04:13:20,100000.0,100000.0


In [16]:
generate_vessel_gantt_chart(df_eventtable)

The inspection of the logbook data shows that the mission of the Vessel, resulted in four cycles where loading - sailing - unloading - sailing were executed in a sequence. The sites had a capacity of 100, and the Vessel a capacity of 30. It can be seen that the last cycle the loading and unloading take less time because the Vessel sailed only partially filled.